# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


# Data

In [ ]:
df = pd.read_csv('./data/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Data Preprocessing

In [ ]:
X = df.drop('Class', axis=1).values
y = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train = torch.FloatTensor(X_train).to(device)
X_test = torch.FloatTensor(X_test).to(device)
y_train = torch.FloatTensor(y_train).to(device)
y_test = torch.FloatTensor(y_test).to(device)

n_samples = len(y_train)
n_fraud = (y_train == 1).sum().item()
n_normal = (y_train == 0).sum().item()

class_weight_normal = n_samples / (2 * n_normal)
class_weight_fraud = n_samples / (2 * n_fraud)

print(f"Total samples: {n_samples}")
print(f"Normal transactions: {n_normal}")
print(f"Fraudulent transactions: {n_fraud}")

Total samples: 227845
Normal transactions: 227451
Fraudulent transactions: 394


## Create DataLoader

In [ ]:
class FraudDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

train_dataset = FraudDataset(X_train_split, y_train_split)
val_dataset = FraudDataset(X_val, y_val)
test_dataset = FraudDataset(X_test, y_test)

BATCH_SIZE = 2048
NUM_WORKERS = 0

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train DataLoader: {len(train_dataloader)} batches")
print(f"Val DataLoader: {len(val_dataloader)} batches")
print(f"Test DataLoader: {len(test_dataloader)} batches")

Train DataLoader: 84 batches
Val DataLoader: 28 batches
Test DataLoader: 28 batches


# Model

In [ ]:
class FraudDetectionNet(nn.Module):
    def __init__(self, input_features=30):
        super(FraudDetectionNet, self).__init__()
        
        # Hidden layers with ReLU activation
        self.fc1 = nn.Linear(input_features, 64)
        self.fc2 = nn.Linear(64, 32)
        
        # Output layer
        self.fc3 = nn.Linear(32, 1)
        
        # Activation functions
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

## Training Setup

In [ ]:
pos_weight = torch.tensor([class_weight_fraud / class_weight_normal]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Loss Function: Binary Cross-Entropy with class weighting")
print(f"Positive weight (fraud): {pos_weight.item():.4f}")

Loss Function: Binary Cross-Entropy with class weighting
Positive weight (fraud): 577.2868


# Training Loop

In [ ]:
def trainer(model, train_loader, val_loader, num_epochs=20):
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    
    return train_losses, val_losses

print("Starting training...\n")
train_losses, val_losses = trainer(model, train_dataloader, val_dataloader, num_epochs=20)
print("\nTraining complete!")

Starting training...

Epoch [1/20], Train Loss: 0.0720, Val Loss: 0.6837
Epoch [2/20], Train Loss: 0.0612, Val Loss: 0.7284
Epoch [3/20], Train Loss: 0.0551, Val Loss: 0.7675
Epoch [4/20], Train Loss: 0.0499, Val Loss: 0.8287
Epoch [5/20], Train Loss: 0.0475, Val Loss: 0.8941
Epoch [6/20], Train Loss: 0.0460, Val Loss: 0.9815
Epoch [7/20], Train Loss: 0.0409, Val Loss: 0.9493
Epoch [8/20], Train Loss: 0.0416, Val Loss: 1.0210
Epoch [9/20], Train Loss: 0.0384, Val Loss: 1.1221
Epoch [10/20], Train Loss: 0.0347, Val Loss: 1.1917
Epoch [11/20], Train Loss: 0.0317, Val Loss: 1.1939
Epoch [12/20], Train Loss: 0.0289, Val Loss: 1.3261
Epoch [13/20], Train Loss: 0.0276, Val Loss: 1.3223
Epoch [14/20], Train Loss: 0.0266, Val Loss: 1.3304
Epoch [15/20], Train Loss: 0.0251, Val Loss: 1.4357
Epoch [16/20], Train Loss: 0.0247, Val Loss: 1.4597
Epoch [17/20], Train Loss: 0.0256, Val Loss: 1.5383
Epoch [18/20], Train Loss: 0.0227, Val Loss: 1.5323
Epoch [19/20], Train Loss: 0.0217, Val Loss: 1.5656

# Evaluation & Testing

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int).flatten()
            all_probs.extend(probs.flatten())
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    roc_auc = roc_auc_score(all_labels, all_probs)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp
    }

test_metrics = evaluate(model, test_dataloader, device)

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1 Score:  {test_metrics['f1']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")
print("\nConfusion Matrix:")
print(f"True Negatives:  {test_metrics['tn']}")
print(f"False Positives: {test_metrics['fp']}")
print(f"False Negatives: {test_metrics['fn']}")
print(f"True Positives:  {test_metrics['tp']}")

TEST SET RESULTS
Precision: 0.2410
Recall:    0.8878
F1 Score:  0.3791
ROC-AUC:   0.9657

Confusion Matrix:
True Negatives:  56590
False Positives: 274
False Negatives: 11
True Positives:  87
